# Trabajo con datos textuales. 

In [2]:
# cagar los datos
from sklearn.datasets import fetch_20newsgroups
# dataset con alrededor de 18.846 mensaje asigndos
# enter 20 grupos. 

# Las sigiuentes son las categorías. 
categories = [
    "alt.atheism",
    "soc.religion.christian",
    "comp.graphics",
    "sci.med",
]

#mezclar todos los datos. 
twenty_train = fetch_20newsgroups(
    subset="train",
    categories=categories,
    shuffle=True,
    random_state=42,
)

/home/mgtdev/documentos_estudio/ESTADISTICA/ANALITICA/.venv/lib/python3.14/site-packages/sklearn/datasets/_base.py:1521: UserWarning: Retry downloading from url: https://ndownloader.figshare.com/files/5975967
  warnings.warn(f"Retry downloading from url: {remote.url}")


In [3]:
twenty_train.target_names

['alt.atheism', 'comp.graphics', 'sci.med', 'soc.religion.christian']

In [4]:
len(twenty_train.data)

2257

In [5]:
twenty_train.filenames[0:10]

array(['/home/mgtdev/scikit_learn_data/20news_home/20news-bydate-train/comp.graphics/38440',
       '/home/mgtdev/scikit_learn_data/20news_home/20news-bydate-train/comp.graphics/38479',
       '/home/mgtdev/scikit_learn_data/20news_home/20news-bydate-train/soc.religion.christian/20737',
       '/home/mgtdev/scikit_learn_data/20news_home/20news-bydate-train/soc.religion.christian/20942',
       '/home/mgtdev/scikit_learn_data/20news_home/20news-bydate-train/soc.religion.christian/20487',
       '/home/mgtdev/scikit_learn_data/20news_home/20news-bydate-train/soc.religion.christian/20891',
       '/home/mgtdev/scikit_learn_data/20news_home/20news-bydate-train/soc.religion.christian/20914',
       '/home/mgtdev/scikit_learn_data/20news_home/20news-bydate-train/sci.med/58110',
       '/home/mgtdev/scikit_learn_data/20news_home/20news-bydate-train/sci.med/58114',
       '/home/mgtdev/scikit_learn_data/20news_home/20news-bydate-train/sci.med/58838'],
      dtype='<U93')

In [6]:
len(twenty_train.filenames)

2257

In [7]:
# ejemplo de uno de los mensajes
print("\n".join(twenty_train.data[0].split("\n")[:10]))

From: sd345@city.ac.uk (Michael Collier)
Subject: Converting images to HP LaserJet III?
Nntp-Posting-Host: hampton
Organization: The City University
Lines: 14

Does anyone know of a good way (standard PC application/PD utility) to
convert tif/img/tga files into LaserJet III format.  We would also like to
do the same, converting to HPGL (HP plotter) files.



In [8]:
# lo que queremos hacer es ver como se clasifica
# el mensaje. 
print(twenty_train.target_names[twenty_train.target[0]])

comp.graphics


In [ ]:
# primeros 10 clasificaciones. 
twenty_train.target[:10]

array([1, 1, 3, 3, 3, 3, 3, 2, 2, 2])

In [10]:
# ver el nombre correspondiente. 
for t in twenty_train.target[:10]:
    print(twenty_train.target_names[t])

comp.graphics
comp.graphics
soc.religion.christian
soc.religion.christian
soc.religion.christian
soc.religion.christian
soc.religion.christian
sci.med
sci.med
sci.med


Ahora veamos los métodos de sickit-learn para la clasificacion de texto. 

## Extracción de características de los archivos de texto

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
# El proposito es transormar colecciones de texto no 
# estructurado en vectore numéricos de caracter´siticas
# basados en la frecuencia con la que aparece cada palabra tokenizada. 



# instancia de count vectorizer, es le método 
# que permite vectorizar las palabras. 
count_vect = CountVectorizer()

# aplicar ajuste y transformar conjunto de datos. 
# Crear matriz término documento y hacer los respectivos
# conteos. Con .fit() se recorre la lista, aplica limpieza
# con _transform() se crea la matriz. 
X_train_counts = count_vect.fit_transform(twenty_train.data)

# ver las dimensiones. 
X_train_counts.shape

# Esta vienene a ser la matríz término documento
# donde las filas son los documentos y las columnas
# son las palabras de todos los documentos y la matriz
# se llena con 0 y 1 identificando si la palabra
# está en el respectivo doccumento o no. 

(2257, 35788)

In [12]:
# El componente vocabulario contiene el nombre de las columnas
# y por ende, nos permite saber el número de la columna
# a la que pertenece una palabra específica. 
count_vect.vocabulary_.get("algorithm")

4690

In [ ]:
from sklearn.feature_extraction.text import TfidfTransformer

# aplicar transformador TF-IDF para controlar las stop-words
# y solamente dejar las palabras importantes. 

# instancia, indicando que use el inverso de la distancia
# para la transformación. 
tf_transformer = TfidfTransformer(use_idf=False).fit(X_train_counts)

# Hacer la transformación en el data set origninal. 
X_train_tf = tf_transformer.transform(X_train_counts)

# ver las dimensiones. 
X_train_tf.shape

# pero ahora es un matriz de punto-flotatnte que cambia el peso de las palabras. 

(2257, 35788)

In [14]:
X_train_tf

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 365886 stored elements and shape (2257, 35788)>

In [ ]:
# instancia del método de TF-IDF
tfidf_transformer = TfidfTransformer()

# Creación de matriz TF-IDF 
X_train_tfidf = tfidf_transformer.fit_transform(X_train_counts)
X_train_tfidf.shape

(2257, 35788)

## Entrenamiento del clasificador. 
Para tomar un texto y clasificarlo en uno de las posibles categorías. 

In [24]:
from sklearn.naive_bayes import MultinomialNB

# instancia del modelo multinomial
clf = MultinomialNB().fit(X_train_tfidf, twenty_train.target)

In [29]:
# definir los documentos nuevos
docs_new = ["God is love", "OpenGL on the GPU is fast"]

# Crear matriz término documento de los documentos
X_new_counts = count_vect.transform(docs_new)

# Crear matriz TF-IDF
X_new_tfidf = tfidf_transformer.transform(X_new_counts)

# Hacer predicciones con le método de naive_bayes multinomial
predicted = clf.predict(X_new_tfidf)

#
for doc, category in zip(docs_new, predicted):
    print("%r => %s" % (doc, twenty_train.target_names[category]))

'God is love' => soc.religion.christian
'OpenGL on the GPU is fast' => comp.graphics


## Construcción de un pipeline

In [31]:
from sklearn.pipeline import Pipeline

text_clf = Pipeline(
    [
        ("vect", CountVectorizer()), # creación matriz término documento
        ("tfidf", TfidfTransformer()), # creación matriz TF-IDF
        ("clf", MultinomialNB()), # clasificador con naive bayes
    ]
)

In [32]:
text_clf.fit(twenty_train.data, twenty_train.target)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('vect', ...), ('tfidf', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](4,)","[0,1,2,3]"
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True


## Evaluación del desempeño sobre el conjunto de prueba

In [ ]:
import numpy as np

#traemos la parte de test
twenty_test = fetch_20newsgroups(
    subset="test", # que sean los datos de test
    categories=categories, # las mismas categorías
    shuffle=True, # aleatorizar
    random_state=42, # usar la misma semilla
    #para que sean los correspondientes
    # con los datos de entrenamiento. 

)

# documetnos de prueba
docs_test = twenty_test.data

# predicción
predicted = text_clf.predict(docs_test)

# proporción de los que son iguales. 
np.mean(predicted == twenty_test.target)

np.float64(0.8348868175765646)

In [34]:
from sklearn.linear_model import SGDClassifier

#también se pueden usar otro clasificadores
# en este caso, clasificador de gradiente
# descendente estocástico. 

text_clf = Pipeline(
    [
        ("vect", CountVectorizer()),
        ("tfidf", TfidfTransformer()),
        (
            "clf",
            SGDClassifier(
                loss="hinge",
                penalty="l2",
                alpha=1e-3,
                random_state=42,
                max_iter=5,
                tol=None,
            ),
        ),
    ]
)

# creación de matriz termino-documetos, creación 
# de matriz TF-IDF y clasificador
text_clf.fit(twenty_train.data, twenty_train.target)

# predicciones
predicted = text_clf.predict(docs_test)#

# proporción
np.mean(predicted == twenty_test.target)

np.float64(0.9101198402130493)

In [35]:
from sklearn import metrics
# esto es un método que me da todo un reporte de un clasificador

metrics.classification_report(
    twenty_test.target, predicted, target_names=twenty_test.target_names
)

'                        precision    recall  f1-score   support\n\n           alt.atheism       0.95      0.80      0.87       319\n         comp.graphics       0.87      0.98      0.92       389\n               sci.med       0.94      0.89      0.91       396\nsoc.religion.christian       0.90      0.95      0.93       398\n\n              accuracy                           0.91      1502\n             macro avg       0.91      0.91      0.91      1502\n          weighted avg       0.91      0.91      0.91      1502\n'

In [36]:
# también tiene la matriz de confusión 
metrics.confusion_matrix(twenty_test.target, predicted)

array([[256,  11,  16,  36],
       [  4, 380,   3,   2],
       [  5,  35, 353,   3],
       [  5,  11,   4, 378]])

## Afinamiento de parámetros usando grid search

In [ ]:
from sklearn.model_selection import GridSearchCV

# método para encontrar la combinación óptima de parámetros

parameters = {
    # parámetros y sus posbiles valores
    "vect__ngram_range": [(1, 1), (1, 2)], # count vectorizer
    "tfidf__use_idf": (True, False), # creación de TF-IDF
    "clf__alpha": (1e-2, 1e-3),# Clasificador
}

# Ojo, el nombre de la componente está al inicio
# vect
# tfidf
# clf 
# Y el nombre del parámetro está separado por 2 guiones
# bajos
# __ngram_range
#__use_idf
#__alpha

# Hay que tener en cuenta que esta es una técnica
# que pued tomar bastante tiempo ya que necesita
# hacer todas las combinaciones de los parámetros

In [38]:
# le pasamos los datos, los parámetros, k-fold cross valitadion con 5 grupos
# y sin paralelizar. 
gs_clf = GridSearchCV(text_clf, parameters, cv=5, n_jobs=-1,)

In [39]:
# entrenamiento y ajuste con los datos de entreamiento. 
gs_clf = gs_clf.fit(twenty_train.data[:400], twenty_train.target[:400],)

In [40]:
# que nos devuelva la clase a la que pertenece. 
twenty_train.target_names[gs_clf.predict(["God is love"])[0]]

'soc.religion.christian'

In [41]:
gs_clf.best_score_

np.float64(0.9175000000000001)

In [42]:
# parámetros que estimo para los hiper-parámetros
for param_name in sorted(parameters.keys()):
    print("%s: %r" % (param_name, gs_clf.best_params_[param_name]))

clf__alpha: 0.001
tfidf__use_idf: True
vect__ngram_range: (1, 1)
